# Self-supervised vision — learn image features without labels, then probe



_Capstones_



**Three contrastive papers, one system: pretrain on unlabelled images, freeze, and let a linear probe beat a from-scratch net when labels are scarce.**



---



This notebook builds the system one step at a time: check the NT-Xent loss by hand, define the encoder and projection head, pretrain on unlabelled MNIST, freeze, and finally pit a linear probe against a from-scratch net across label budgets. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.

import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Sanity-check the NT-Xent loss by hand

Before any training, we verify the contrastive loss on a tiny worked example. NT-Xent treats one view as the *anchor* and its augmented partner as the only *positive*; every other sample in the batch is a negative. For unit vectors the dot product is the cosine similarity, so we scale the similarities by `1/tau`, softmax, and read off the probability mass on the true partner. The expected values (`p_12 = 0.7919`, loss `0.2333`) confirm our understanding is right.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Sanity-check NT-Xent on the worked example (i=1, j=2; tau=0.5).
zc = torch.tensor([[1.0, 0.0],   # z1  anchor i
                   [0.8, 0.6],   # z2  positive partner j
                   [-0.6, 0.8],  # z3  negative
                   [0.0, -1.0]]) # z4  negative
tau = 0.5

sims = zc[0] @ zc.t()                       # sim(z1, z_k) for all k (unit vecs -> dot = cosine)
exps = torch.exp(sims / tau)
p_12 = exps[1] / exps[1:].sum()             # softmax prob of true partner (drop self term k=1)
print("NT-Xent check:  p_12 =", round(p_12.item(), 4),
      " loss_12 =", round((-torch.log(p_12)).item(), 4))   # expect 0.7919 / 0.2333

### Step 2 — Define the encoder, projection head, and batched NT-Xent

The **encoder** `f` is a small conv net producing a representation `h` — this is the part we keep and probe later. The **projection head** `g` maps `h` to the space `z` where the loss lives (discarded after pretraining). The batched `nt_xent` takes the `2N` stacked projections of two views, normalises them, builds the full `2N x 2N` similarity matrix, blanks the diagonal (no self-matches), and uses cross-entropy where each row's target is its augmented partner.

In [ ]:
class Encoder(nn.Module):                   # small conv net -> representation h (the thing we keep)
    def __init__(self, feat=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.fc = nn.Linear(32, feat)
    def forward(self, x):
        return F.relu(self.fc(self.net(x)))             # h

class ProjHead(nn.Module):                  # MLP, one hidden layer + ReLU:  z = W2 sigma(W1 h)
    def __init__(self, fin=64, hid=64, out=32):
        super().__init__()
        self.l1 = nn.Linear(fin, hid)
        self.l2 = nn.Linear(hid, out)
    def forward(self, h):
        return self.l2(F.relu(self.l1(h)))             # z (loss lives here)

# The NT-Xent contrastive loss. z is the 2N stacked projections [v1 ; v2].
def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1)               # cosine similarity needs unit vectors
    N = z.shape[0] // 2
    sim = z @ z.t() / tau                    # 2N x 2N scaled cosine-similarity matrix
    sim.fill_diagonal_(-1e9)                 # indicator 1[k != i]: drop the self term
    targets = torch.cat([torch.arange(N) + N, torch.arange(N)]).to(z.device)  # partner of row i
    return F.cross_entropy(sim, targets)     # softmax over similarities = NT-Xent

### Step 3 — Pretrain SimCLR on unlabelled MNIST

We define a stochastic two-view augmentation (random crop + occasional blur) and grab a 3000-image MNIST subset. The labels are loaded but used **only later** for the probe — pretraining never sees them. Each step makes two augmented views of the same batch, encodes and projects them, and minimises NT-Xent so the two views of an image are pulled together while different images are pushed apart.

In [ ]:
# Two-view augmentation + an UNLABELLED MNIST subset (torchvision, preinstalled).
aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5, 1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]
labels = torch.tensor([raw[i][1] for i in idx])    # used ONLY for the probe, never in pretraining

# Pretrain SimCLR (NO labels).
enc, proj = Encoder().to(device), ProjHead().to(device)
opt = torch.optim.Adam(list(enc.parameters()) + list(proj.parameters()), lr=1e-3)
enc.train(); proj.train()
B = 128
for ep in range(15):
    perm = np.random.permutation(len(imgs))
    tot = 0.0
    nb = 0
    for s in range(0, len(imgs), B):
        bi = perm[s:s + B]
        v1 = torch.stack([aug(imgs[i]) for i in bi]).to(device)   # view 1
        v2 = torch.stack([aug(imgs[i]) for i in bi]).to(device)   # view 2
        loss = nt_xent(proj(enc(torch.cat([v1, v2]))))
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item(); nb += 1
    if ep % 3 == 0:
        print(f"  pretrain ep {ep}  NT-Xent loss {tot/nb:.3f}")

### Step 4 — Freeze, then probe vs. from-scratch across label budgets

Following the linear-evaluation protocol, we **freeze** the encoder and extract features `h` once. `linear_probe` trains only a linear classifier on those frozen features; `from_scratch` trains a whole fresh conv net end-to-end on the same handful of labels. Sweeping the label budget shows the headline result: the frozen-SimCLR probe wins at every budget, and the gap is **largest when labels are scarcest** — exactly when good pretrained features matter most.

In [ ]:
# FREEZE the encoder; extract features h (linear-evaluation protocol).
enc.eval()
with torch.no_grad():
    feats = enc(torch.stack([base(im) for im in imgs]).to(device)).cpu()

def linear_probe(n_lab):                     # train ONLY a linear classifier on frozen h
    accs = []
    for seed in range(3):
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        clf = nn.Linear(feats.shape[1], 10)
        o = torch.optim.Adam(clf.parameters(), lr=0.05)
        for _ in range(200):
            o.zero_grad()
            F.cross_entropy(clf(feats[tr]), labels[tr]).backward()
            o.step()
        with torch.no_grad():
            accs.append((clf(feats[te]).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

def from_scratch(n_lab):                      # train a fresh conv net end-to-end on the few labels
    accs = []
    for seed in range(3):
        torch.manual_seed(seed)
        g = np.random.RandomState(seed)
        tr = g.permutation(len(labels))[:n_lab]
        te = g.permutation(len(labels))[-600:]
        net = nn.Sequential(Encoder(), nn.Linear(64, 10))
        o = torch.optim.Adam(net.parameters(), lr=1e-3)
        Xtr = torch.stack([base(imgs[i]) for i in tr])
        net.train()
        for _ in range(60):
            o.zero_grad()
            F.cross_entropy(net(Xtr), labels[tr]).backward()
            o.step()
        net.eval()
        with torch.no_grad():
            Xte = torch.stack([base(imgs[i]) for i in te])
            accs.append((net(Xte).argmax(1) == labels[te]).float().mean().item())
    return float(np.mean(accs))

# The headline: probe (frozen, no-label pretrain) vs from-scratch, across label budgets.
print("\nlabels | probe(frozen SimCLR) | from-scratch | probe wins?")
for n in [20, 50, 100, 300]:
    p, s = linear_probe(n), from_scratch(n)
    print(f"{n:6d} |        {p:.3f}         |    {s:.3f}     |   {'YES' if p > s else 'no'}")
# The frozen-SimCLR linear probe beats from-scratch at every budget -- biggest gap when labels are fewest.
# (Exact numbers vary by hardware/seed; this is our small run, not the papers' reported results.)

## Visualize the data & results

_Does the finished system work: in the low-label regime, does a linear probe on frozen contrastive features beat a from-scratch classifier?_

### Step 1 — Redefine the pieces and pretrain (condensed)

This visualization cell is a self-contained rerun of the capstone. We re-import, re-seed, and redefine the same `Encoder`, `ProjHead`, and `nt_xent`, then pretrain the SimCLR encoder on the same 3000-image unlabelled MNIST subset — identical logic to the reference build, just gathered here so the comparison below stands alone.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
import torchvision, torchvision.transforms as T
torch.manual_seed(0); np.random.seed(0)

# Finished capstone, condensed: pretrain a tiny SimCLR encoder on UNLABELLED MNIST, freeze it,
# then compare a linear probe on its frozen features vs a from-scratch net across label budgets.
class Encoder(nn.Module):
    def __init__(s, feat=64):
        super().__init__()
        s.net = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                              nn.AdaptiveAvgPool2d(1), nn.Flatten())
        s.fc = nn.Linear(32, feat)
    def forward(s, x): return F.relu(s.fc(s.net(x)))
class ProjHead(nn.Module):
    def __init__(s, fin=64, hid=64, out=32):
        super().__init__(); s.l1=nn.Linear(fin,hid); s.l2=nn.Linear(hid,out)
    def forward(s, h): return s.l2(F.relu(s.l1(h)))

def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1); N = z.shape[0]//2
    sim = z @ z.t() / tau; sim.fill_diagonal_(-1e9)
    tgt = torch.cat([torch.arange(N)+N, torch.arange(N)])
    return F.cross_entropy(sim, tgt)

aug  = T.Compose([T.RandomResizedCrop(28, scale=(0.5,1.0)),
                  T.RandomApply([T.GaussianBlur(3)], 0.5), T.ToTensor()])
base = T.ToTensor()
raw  = torchvision.datasets.MNIST("./data", train=True, download=True)
idx  = np.random.RandomState(0).permutation(len(raw))[:3000]
imgs = [raw[i][0] for i in idx]; labels = torch.tensor([raw[i][1] for i in idx])

enc, proj = Encoder(), ProjHead()
opt = torch.optim.Adam(list(enc.parameters())+list(proj.parameters()), lr=1e-3)
enc.train(); proj.train(); B=128
for ep in range(15):
    perm = np.random.permutation(len(imgs))
    for s0 in range(0, len(imgs), B):
        bi = perm[s0:s0+B]
        v1 = torch.stack([aug(imgs[i]) for i in bi]); v2 = torch.stack([aug(imgs[i]) for i in bi])
        loss = nt_xent(proj(enc(torch.cat([v1, v2]))))
        opt.zero_grad(); loss.backward(); opt.step()

### Step 2 — Freeze, then compare probe vs. from-scratch

We freeze the encoder, extract features once, and define the same `probe` and `scratch` evaluators. Printing accuracy for each label budget reproduces the headline finding: the linear probe on frozen contrastive features beats the from-scratch net at every budget, with the largest margin at the fewest labels.

In [ ]:
enc.eval()
with torch.no_grad():
    feats = enc(torch.stack([base(im) for im in imgs]))

def probe(n):
    a=[]
    for seed in range(3):
        g=np.random.RandomState(seed); tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        clf=nn.Linear(feats.shape[1],10); o=torch.optim.Adam(clf.parameters(),lr=0.05)
        for _ in range(200): o.zero_grad(); F.cross_entropy(clf(feats[tr]),labels[tr]).backward(); o.step()
        with torch.no_grad(): a.append((clf(feats[te]).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)
def scratch(n):
    a=[]
    for seed in range(3):
        torch.manual_seed(seed); g=np.random.RandomState(seed)
        tr=g.permutation(len(labels))[:n]; te=g.permutation(len(labels))[-600:]
        net=nn.Sequential(Encoder(), nn.Linear(64,10)); o=torch.optim.Adam(net.parameters(),lr=1e-3)
        Xtr=torch.stack([base(imgs[i]) for i in tr]); net.train()
        for _ in range(60): o.zero_grad(); F.cross_entropy(net(Xtr),labels[tr]).backward(); o.step()
        net.eval()
        with torch.no_grad():
            Xte=torch.stack([base(imgs[i]) for i in te]); a.append((net(Xte).argmax(1)==labels[te]).float().mean().item())
    return round(float(np.mean(a)),3)

for n in [20,50,100,300]:
    print(n, "probe", probe(n), "scratch", scratch(n))
# probe (frozen contrastive features) > from-scratch at every budget; biggest gap at the fewest labels.